In [1]:
import pandas as pd

df= pd.read_csv(r"D:\Course\python\energy forecasting\data\processed\final_energy_forecasting_dataset.csv")


In [2]:
df.columns

Index(['time', 'load', 'solar', 'wind', 'wind_onshore', 'wind_offshore',
       'hour', 'day_of_week', 'day_of_month', 'month', 'is_weekend',
       'is_holiday', 'load_forecast', 'temperature', 'humidity', 'wind_speed',
       'precipitation', 'weather_code', 'temperature_lag_1', 'humidity_lag_1',
       'temperature_lag_24', 'humidity_lag_24', 'load_lag_1', 'load_lag_24',
       'load_lag_168', 'rolling_mean_24', 'rolling_std_24'],
      dtype='object')

In [4]:
print(df["load"].mean())


55506.61274265804


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 50225 entries, 0 to 50224
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   time                50225 non-null  str    
 1   load                50225 non-null  float64
 2   solar               50225 non-null  float64
 3   wind                50225 non-null  float64
 4   wind_onshore        50225 non-null  float64
 5   wind_offshore       50225 non-null  float64
 6   hour                50225 non-null  int64  
 7   day_of_week         50225 non-null  int64  
 8   day_of_month        50225 non-null  int64  
 9   month               50225 non-null  int64  
 10  is_weekend          50225 non-null  int64  
 11  is_holiday          50225 non-null  int64  
 12  load_forecast       50225 non-null  float64
 13  temperature         50225 non-null  float64
 14  humidity            50225 non-null  int64  
 15  wind_speed          50225 non-null  float64
 16  precipitation  

In [15]:
import pandas as pd

df = pd.read_csv("D:/Course/python/energy forecasting/data/processed/final_energy_forecasting_dataset.csv")

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")

# create time index
df["time_idx"] = (df["time"] - df["time"].min()).dt.total_seconds() // 3600

# 🔥 ADD THIS LINE HERE
df["time_idx"] = df["time_idx"].astype(int)

# single series
df["series_id"] = 0


In [16]:
features_known = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday"
]

features_unknown = [
    "load",
    "solar",
    "wind",
    "temperature",
    "humidity"
]


In [17]:
from pytorch_forecasting import TimeSeriesDataSet

training = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="load",
    group_ids=["series_id"],
    max_encoder_length=168,
    max_prediction_length=24,
    time_varying_known_reals=features_known,
    time_varying_unknown_reals=features_unknown,
)


In [18]:
df["time_idx"] = df["time_idx"].astype(int)


In [19]:
train_dataloader = training.to_dataloader(train=True, batch_size=64)


In [20]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    loss=QuantileLoss(),
)


d:\Course\python\energy forecasting\energy_env\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
d:\Course\python\energy forecasting\energy_env\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [21]:
from lightning.pytorch import Trainer

trainer = Trainer(
    max_epochs=20,
    accelerator="auto"  # uses GPU if available
)

trainer.fit(tft, train_dataloader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
d:\Course\python\energy forecasting\energy_env\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoin

Epoch 1:   0%|          | 1/781 [00:00<01:48,  7.18it/s, v_num=0, train_loss_step=894.0, train_loss_epoch=1.08e+3]  

d:\Course\python\energy forecasting\energy_env\Lib\site-packages\lightning\pytorch\loops\training_epoch_loop.py:500: ReduceLROnPlateau conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.


Epoch 19: 100%|██████████| 781/781 [01:46<00:00,  7.33it/s, v_num=0, train_loss_step=273.0, train_loss_epoch=280.0] 

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 781/781 [01:46<00:00,  7.32it/s, v_num=0, train_loss_step=273.0, train_loss_epoch=280.0]


In [22]:
training_cutoff = df["time_idx"].max() - 24*7  # last 7 days for validation


In [23]:
training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="load",
    group_ids=["series_id"],
    max_encoder_length=168,
    max_prediction_length=24,
    time_varying_known_reals=features_known,
    time_varying_unknown_reals=features_unknown,
)


In [24]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    df,
    predict=True,
    stop_randomization=True
)


In [27]:
num_workers = 0


In [28]:
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=64,
    num_workers=0
)

val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=64,
    num_workers=0
)


In [26]:
raw_predictions, x = tft.predict(val_dataloader, mode="raw", return_x=True)

interpretation = tft.interpret_output(raw_predictions, reduction="sum")
tft.plot_interpretation(interpretation)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\Course\python\energy forecasting\energy_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.


RuntimeError: DataLoader worker (pid(s) 24360, 9980, 5604, 7644) exited unexpectedly